# Find Best Model


In [ ]:
from types import SimpleNamespace
import os
import glob
import numpy as np
import pandas as pd
import plotly.express as px


from meta import load_meta

## Args


### Demo


In [ ]:
# args = SimpleNamespace(
#     # io
#     logs_path="data/logs",
#     experiment_name="demo_experiment",
# )

### laser-pulse-shaping-astra-sim


In [ ]:
# args = SimpleNamespace(
#     # io
#     data_path="data/laser-pulse-shaping-astra-sim-v11-ln-emit-normalized",
#     logs_path="data/logs",
#     experiment_name="laser-pulse-shaping-astra-sim-v11-ln-emit-normalized",  # use .old for previous version
# )

args = SimpleNamespace(
    # io
    data_path="data/laser-pulse-shaping-astra-sim-v20+v30-ln-std-emit-normalized-outliers-removed",
    logs_path="data/logs",
    experiment_name="laser-pulse-shaping-astra-sim-v20+v30-ln-std-emit-normalized-outliers-removed",
)

In [ ]:
rows: list[dict] = []

for run_path in glob.glob(os.path.join(args.logs_path, args.experiment_name, "*")):
    print(f"Loading meta from {run_path}")
    meta = load_meta(run_path)
    rows.append(meta)

table = pd.DataFrame(rows)

assert all(table["data_path"] == args.data_path)

table["max_width"] = table["hidden_layer_sizes"].apply(lambda x: max(x))
table["depth"] = table["hidden_layer_sizes"].apply(lambda x: len(x))


table["best_val_train_loss_ratio"] = table["best_val_loss"] / table["best_train_loss"]

table["log10(trainable_params)"] = table["trainable_params"].apply(
    lambda x: np.log10(x)
)
table["log10(best_val_loss)"] = table["best_val_loss"].apply(lambda x: np.log10(x))
table["log10(best_train_loss)"] = table["best_train_loss"].apply(lambda x: np.log10(x))
table["log10(learning_rate)"] = table["learning_rate"].apply(lambda x: np.log10(x))
table["log2(batch_size)"] = table["batch_size"].apply(lambda x: np.log2(x))
table["log10(best_step)"] = table["best_step"].apply(lambda x: np.log10(x))
table["log10(trainable_params)"] = table["trainable_params"].apply(
    lambda x: np.log10(x)
)


table

In [ ]:
px.scatter_3d(
    table,
    x="max_width",
    y="depth",
    z="log10(best_val_loss)",
    color="log2(batch_size)",
).show()

px.scatter_3d(
    table,
    x="max_width",
    y="depth",
    z="log10(best_val_loss)",
    color="best_val_train_loss_ratio",
    color_continuous_scale="RdBu",  # red–blue color scale
    range_color=[0, 2],  # set colorbar limits
).show()

px.scatter_3d(
    table,
    x="max_width",
    y="depth",
    z="log10(best_val_loss)",
    color="log10(learning_rate)",
).show()

px.scatter(
    table,
    x="max_width",
    y="log10(best_val_loss)",
    color="log10(learning_rate)",
).show()

px.scatter(
    table,
    x="depth",
    y="log10(best_val_loss)",
    color="log10(learning_rate)",
).show()

## Best Model Overall


### Paretro front


In [ ]:
px.scatter(
    table,
    x="best_val_loss",
    y="best_val_train_loss_ratio",
    hover_name="version",
    log_x=True,
    log_y=True,
    color="max_width",
).show()

In [ ]:
px.scatter(
    table,
    x="best_val_loss",
    y="best_val_train_loss_ratio",
    hover_name="version",
    log_x=True,
    log_y=True,
    color="log10(trainable_params)",
).show()

In [ ]:
px.scatter(
    table,
    x="best_val_loss",
    y="best_val_train_loss_ratio",
    hover_name="version",
    log_x=True,
    log_y=True,
    color="log10(learning_rate)",
).show()

In [ ]:
px.scatter(
    table,
    x="best_val_loss",
    y="best_val_train_loss_ratio",
    hover_name="version",
    log_x=True,
    log_y=True,
    color="depth",
).show()

In [ ]:
px.scatter(
    table,
    x="best_val_loss",
    y="best_val_train_loss_ratio",
    hover_name="version",
    log_x=True,
    log_y=True,
    color="log10(trainable_params)",
).show()

In [ ]:
px.scatter(
    table,
    x="best_val_loss",
    y="best_val_train_loss_ratio",
    hover_name="version",
    log_x=True,
    log_y=True,
    color="log10(best_step)",
).show()

### Select


Logbook:

- `v11`
  - 2026-01-09-16-13-59-NNYpe2iL:

    `hidden_layers=[50]**4`
    `best_val_loss=0.0017`

    -->`best_val_train_loss_ratio=1.9`<--

    ```python
      def select_best_model():
          df = table.copy()
          return df.iloc[df["log10(best_val_loss)"].argmin()]
    ```

  - 2026-01-09-16-13-58-hGpjScbq:

    -->`hidden_layers=[50]`<--
    `best_val_loss=0.0055`

    ```python
    def select_best_model():
        df = table.copy()
        df = df[df["best_val_train_loss_ratio"] <= 1.2]
        return df.iloc[df["log10(best_val_loss)"].argmin()]
    ```

- `v20`
  - 2026-01-13
  - 2026-02-07


In [ ]:
def select_best_model():
    df = table.copy()
    # df = df[df["best_val_train_loss_ratio"] <= 1.2]
    return df.iloc[df["log10(best_val_loss)"].argmin()]


best = select_best_model()
best,

In [ ]:
best_path = f"{args.logs_path}/{args.experiment_name}/_best"

if os.path.islink(best_path):
    os.unlink(best_path)

os.symlink(best["version"], best_path, target_is_directory=True)

In [ ]:
selected_version = best["version"]
selected_version